# Laboratorio 3. Analisis exploratorio del dataset ASL Alphabet

Fabian Prado Dluzniewski 23427

Abby Donis 22440

Hansel Lopez 19026

Este notebook cubre los ejercicios 1 y 2. El objetivo es entender el dataset
antes de modelar, sabiendo que la meta final es reconocer la letra del alfabeto
ASL que aparece en una fotografia de una mano.

Las preguntas que guian el analisis son las siguientes:

1. Que resolucion y formato tienen las imagenes y son homogeneos.
2. Esta balanceado el dataset entre las 29 clases.
3. Cuanta variabilidad hay dentro de una misma letra.
4. Que letras se parecen visualmente entre si y podrian confundir al modelo.
5. Son independientes las imagenes de una misma clase o hay redundancia.
6. Como conviene construir los conjuntos de entrenamiento, validacion y prueba.

### IMPORTS

In [1]:
import logging
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

import kagglehub
from kagglesdk import KaggleClient
from kagglesdk.datasets.types.dataset_api_service import ApiListTreeDatasetFilesRequest

logging.getLogger("kagglehub").setLevel(logging.CRITICAL)
plt.style.use("seaborn-v0_8-whitegrid")

### CONFIGURACION

In [2]:
DATASET = "grassknoted/asl-alphabet"
SUB_TRAIN = "asl_alphabet_train/asl_alphabet_train"

CLASES = [chr(c) for c in range(ord("A"), ord("Z") + 1)] + ["space", "del", "nothing"]
SUBMUESTRA = 600
SEMILLA = 42

DIR_DATOS = Path("./data")
DIR_DATOS.mkdir(exist_ok=True)
RUTA_INVENTARIO = DIR_DATOS / "inventario_clases.csv"

random.seed(SEMILLA)
np.random.seed(SEMILLA)

print(f"clases {len(CLASES)}  submuestra por clase {SUBMUESTRA}")

clases 29  submuestra por clase 600


### ACCESO A LOS DATOS

El dataset pesa cerca de 1 GB y no se versiona. Se accede con la API de Kaggle
a traves de la libreria `kagglehub`, que deja todo en `~/.cache/kagglehub`,
fuera del repositorio.

Vale la pena aclarar un punto que costo trabajo durante el desarrollo.
`dataset_download` acepta un argumento `path` para traer un archivo suelto, y
esa parecia la via para bajar solo la submuestra. En la practica no funciona a
esta escala. Cada llamada con `path` implica al menos dos peticiones a la API,
una para resolver el dataset y otra para el archivo, de modo que 600 imagenes
por clase se convierten en mas de 35,000 peticiones. Kaggle responde 429 y
bloquea la cuenta entera. Se midio el ritmo resultante y era exactamente cero.

Por eso se hace una sola llamada que trae el dataset completo y el
submuestreo se resuelve localmente. Es una peticion en lugar de 35,000.

In [3]:
t0 = time.time()
RAIZ = Path(kagglehub.dataset_download(DATASET))
print(f"dataset disponible en {RAIZ}")
print(f"tiempo {(time.time()-t0)/60:.1f} min")

DIR_TRAIN = RAIZ / SUB_TRAIN
print(f"carpetas de clase encontradas: {len(list(DIR_TRAIN.iterdir()))}")

dataset disponible en /Users/fabianprado/.cache/kagglehub/datasets/grassknoted/asl-alphabet/versions/1
tiempo 0.0 min
carpetas de clase encontradas: 29


### INVENTARIO POR CLASE

`ListTreeDatasetFiles` acepta un `path`, asi que permite consultar una carpeta
de clase a la vez en lugar de recorrer las 87,000 rutas del dataset. La
limitante es que devuelve como maximo 200 nombres por pagina, de modo que una
clase de 3,000 imagenes son 15 paginas y las 29 clases juntas vuelven a sumar
435 peticiones, que es exactamente lo que dispara el bloqueo por 429.

Por eso el conteo completo se hace sobre las carpetas ya descargadas, que da el
mismo resultado sin costo de red, y la API se usa para verificar una clase y
confirmar que el conteo en disco coincide con lo que reporta Kaggle.

In [4]:
def contar_por_api(clase, tam_pagina=200):
    total, token = 0, None
    with KaggleClient() as cli:
        while True:
            pet = ApiListTreeDatasetFilesRequest()
            pet.owner_slug, pet.dataset_slug = DATASET.split("/")
            pet.path = f"{SUB_TRAIN}/{clase}"
            pet.page_size = tam_pagina
            if token:
                pet.page_token = token
            resp = cli.datasets.dataset_api_client.list_tree_dataset_files(pet)
            total += len(resp.files)
            token = getattr(resp, "next_page_token", None)
            if not token:
                return total


def inventario_clases():
    if RUTA_INVENTARIO.exists():
        return pd.read_csv(RUTA_INVENTARIO)

    filas = []
    for clase in CLASES:
        rutas = sorted((DIR_TRAIN / clase).glob("*.jpg"))
        filas.append({
            "clase": clase,
            "imagenes": len(rutas),
            "kb_promedio": np.mean([r.stat().st_size for r in rutas]) / 1024,
        })
    inv = pd.DataFrame(filas)
    inv.to_csv(RUTA_INVENTARIO, index=False)
    return inv


inventario = inventario_clases()
print(f"clases inventariadas: {len(inventario)}")
print(f"total de imagenes: {inventario['imagenes'].sum():,}")

clases inventariadas: 29
total de imagenes: 87,000


In [5]:
en_disco = int(inventario.loc[inventario["clase"] == "A", "imagenes"].iloc[0])
try:
    por_api = contar_por_api("A")
    print(f"clase A segun la API:    {por_api:,}")
    print(f"clase A segun el disco:  {en_disco:,}")
    print("los conteos coinciden" if por_api == en_disco else "los conteos diferen")
except Exception as e:
    print(f"la API no respondio ({str(e)[:60]}), se usa solo el conteo en disco")
    print(f"clase A segun el disco: {en_disco:,}")

clase A segun la API:    3,000
clase A segun el disco:  3,000
los conteos coinciden
